In [ ]:
# Archived experiments: This notebook preserves superseded approaches and
#  debugging history. Some experiments contain known validation problems or 
# depend on local historical checkpoints. It is not the application’s execution 
# or evaluation workflow.

In [ ]:
# Obsolete: replaced with four partition time, replaced from cell 5-6
# ## CHRONOLOGICAL SPLIT AND MODEL TRAINING
# # ^

# # chronological split:
# # setting up input and output variables for training model
# df['elo_difference'] = df['home_elo_pre_match'] - df['away_elo_pre_match']

# def determine_outcome(goal_diff):
#     # np.where handles Pandas Series element-by-element safely
#     return np.where(goal_diff > 0, 'H', np.where(goal_diff < 0, 'A', 'D'))

# goal_difference = df['home_score'] - df['away_score']
# df['match_outcome'] = determine_outcome(goal_difference) # H, A, D; match result

# # Encode text labels into integers (e.g., A=0, D=1, H=2)
# encoder = LabelEncoder()
# df['target'] = encoder.fit_transform(df['match_outcome']) # match result encoded in output

# features = ['elo_difference', 'is_home_team_host', 'neutral', 'form_difference'] #input
# X = df[features]
# y = df['target']

# # split data between training and test data 
# split_date = pd.to_datetime('2022-01-01')
# train_mask = df['date'] < split_date
# test_mask = df['date'] >= split_date


# X_train, y_train = X[train_mask], y[train_mask] # data before 2022
# X_test, y_test = X[test_mask], y[test_mask] # data after 

# #running ML model, gradient boosted trees
# model = XGBClassifier(
#     n_estimators=100,
#     max_depth=4,
#     learning_rate=0.1,
#     objective='multi:softprob', # Outputs probabilities for each class
#     random_state=42
# )
# model.fit(X_train, y_train)

# #evaluating the model
# y_pred = model.predict(X_test)
# y_proba = model.predict_proba(X_test)

# print("Classification Report on Test Data (Post-2022):")
# print(classification_report(y_test, y_pred, target_names=encoder.classes_, zero_division=0))
# loss = log_loss(y_test, y_proba)
# print(f"Multi-class Log Loss: {loss:.4f}")


In [ ]:
# Obsolete: utilizing prev cell's time split for full-data cross-validation
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import TimeSeriesSplit #preventing using future data
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import log_loss, f1_score
# import xgboost as xgb

# # preventing the use of data leakage by using timeSeriesSplit + backtesting (xgboost or logistic regression?)

# # 1. Ensure dataset is sorted chronologically
# df = (
#     df.sort_values("date", kind="stable")
#       .reset_index(drop=True)
# )

# X = df[features]
# y = df['target']  # Target encoded as integers (e.g., 0: Away, 1: Draw, 2: Home)

# # 2. Set up TimeSeriesSplit (e.g., 5 expanding historical folds)
# tscv = TimeSeriesSplit(n_splits=5)

# log_loss_scores = []
# f1_scores = []


# #^model comparison, finding the best model <= ** not needed ** (substitute model comparison cell with this)
# # 3. Evaluate iteratively across time, and compare xgboost and logistic regression 
# # to see that gridsearch is computationally worth it
# ## backtesting
# for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
#     X_fold_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_fold_train, y_val = y.iloc[train_idx], y.iloc[val_idx] #fold is training answers and y_val is validation
    
#     xgb_model = xgb.XGBClassifier(
#         objective='multi:softprob',
#         num_class=3,
#         eval_metric='mlogloss',
#         random_state=42
#     )
#     xgb_model.fit(X_fold_train, y_fold_train)
    
#     # Predict probabilities for multi-class Log Loss & hard classes for F1
#     preds_proba = xgb_model.predict_proba(X_val)
#     preds_class = xgb_model.predict(X_val)
    
#     loss = log_loss(y_val, preds_proba)
#     f1 = f1_score(y_val, preds_class, average='macro')
    
#     log_loss_scores.append(loss)
#     f1_scores.append(f1)
    
#     print(f"Fold {fold + 1} - Validation Log Loss: {loss:.4f} | Macro F1: {f1:.4f}")
    
#     # logistic Regression, used to verify that gridsearch is worth doing (where it is optimiziable)
#     #Scale features strictly on training data to prevent leakage
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_fold_train)
#     X_val_scaled = scaler.transform(X_val)
    
#     # 2. Train Linear Baseline
#     lr_model = LogisticRegression(max_iter=1000)
#     lr_model.fit(X_train_scaled, y_fold_train)
    
#     # 3. Predict & Evaluate
#     lr_preds_proba = lr_model.predict_proba(X_val_scaled)
#     lr_loss = log_loss(y_val, lr_preds_proba)

    
#     # Print both side-by-side inside the loop
#     print(f"Fold {fold + 1} - XGBoost Log Loss: {loss:.4f} | Logistic Reg Log Loss: {lr_loss:.4f}")

# print(f"\nMean Log Loss: {np.mean(log_loss_scores):.4f}")
# print(f"Mean Macro F1: {np.mean(f1_scores):.4f}")

In [ ]:
# Obsolete: original attempt to ensemble, but was later found ineffective
# - overwrites models and replaces dictionary used by newer feature exp
# ## Model Comparison (XGBoost, LightGBM, Random Forest, Logistic Regression)
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import TimeSeriesSplit, cross_val_score
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression

# models = {
#     'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42),
#     'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
#     'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
#     'Logistic Regression': Pipeline([
#         ('scaler', StandardScaler()),
#         ('model', LogisticRegression(max_iter=1000, random_state=42))
#     ])
# }

# # 2. Setup TimeSeriesSplit cross-validation
# tscv = TimeSeriesSplit(n_splits=5)
# results_all_models = []

# # 3. Evaluate each model using negative log loss
# for name, model in models.items():
#     cv_scores = cross_val_score(
#         model, X_train, y_train, 
#         cv=tscv, 
#         scoring='neg_log_loss', 
#         n_jobs=-1
#     )
#     mean_loss = -np.mean(cv_scores)
#     std_loss = np.std(cv_scores)
    
#     results_all_models.append({
#         'Model': name,
#         'Mean Log Loss': mean_loss,
#         'Std Dev': std_loss
#     })

# # Display comparison results sorted by performance
# comparison_df = pd.DataFrame(results_all_models).sort_values(by='Mean Log Loss')
# print("--- Model Comparison Results (Lower Log Loss is Better) ---")
# print(comparison_df.to_string(index=False))





In [ ]:
# Obsolete: Tuning on Gridsearch, part of original ensemble
# ### Hyperparameter Tuning for GridSearch (below below) unneeded as XGBoost shown to be ineffective,
# ### swtiched to ensemble with Tuned LR, XGBoost lightly Tuned (below), LightGMB (no tuning for computational cost)
# xgb_light = XGBClassifier(
#     max_depth=3,            # Shallow trees prevent overfitting
#     learning_rate=0.05,     # Slow, stable learning rate
#     n_estimators=150,       # Moderate number of trees
#     subsample=0.8,          # Uses 80% of rows per tree to increase diversity
#     colsample_bytree=0.8,   # Uses 80% of features per tree
#     eval_metric='mlogloss',
#     random_state=42
# )


# ## Hyperparameter Tuning: GridSearch best model, alters weights on diff components to make ml model best f1,etc value

# weights = compute_sample_weight(
#     class_weight='balanced',
#     y=y_train
# )

# # model_path = 'world_cup_xgb_model.joblib'

# # # if os.path.exists(model_path):
# # #     print(f"Found saved model. Loading '{model_path}'...")
# # #     best_model = joblib.load(model_path)
# # # else:
#     # 1. Define the grid of settings you want to test
# param_grid = {
#     'max_depth': [3, 4, 5],                 # How deep the decision trees go
#     'learning_rate': [0.01, 0.05, 0.1],     # How fast the model learns
#     'n_estimators': [100, 200, 300],        # Number of trees in the forest
#     'subsample': [0.8, 1.0]                 # % of data used per tree (prevents overfitting)
# }

# # 2. Initialize a blank base model
# base_model = XGBClassifier(
#     objective='multi:softprob',
#     random_state=42
# )

# # 3. Set up 
# grid_search = GridSearchCV(
#     estimator=base_model,
#     param_grid=param_grid,
#     scoring='neg_log_loss', 
#     cv=tscv, # time series variable
#     verbose=2 
# )


# # 4. massive testing loop

# # calculates the highest mean_test_score based on the 'accuracy', 'f1_weighted', 'roc_auc_ovr', 'precision'
# # scoring='neg_log_loss' selects the model that minimizes multi-class cross-entropy, negative log loss
# grid_search.fit(X_train, y_train, sample_weight=weights)

# # 5. Extract the winner
# print("Best Parameters found: ", grid_search.best_params_)
# best_model = grid_search.best_estimator_

# joblib.dump(best_model, 'world_cup_xgb_model.joblib') #save model to a file using joblib library
# print("Model saved successfully!")

# # Evaluate the optimized model
# y_pred_opt = best_model.predict(X_test)
# print(classification_report(y_test, y_pred_opt, target_names=encoder.classes_, zero_division=0))




In [ ]:
# Obsolete: tuning LR, part of o ensemble 
# ## Tuning code for Logistic Regression
# from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import numpy as np

# # 1. Create a Pipeline to prevent data leakage during scaling within CV folds
# pipe = Pipeline([
#     ('scaler', StandardScaler()),
#     ('model', LogisticRegression(max_iter=2000, random_state=42))
# ])

# # 2. Hyperparameter space for Logistic Regression
# param_distributions = {
#     'model__C': np.logspace(-3, 2, 50),              # Regularization strength
#     'model__solver': ['saga'],                      # Solver compatible with all penalties
#     'model__l1_ratio': np.linspace(0, 1, 10)        # ElasticNet mixing parameter
# }

# # 3. TimeSeriesSplit cross-validation
# tscv = TimeSeriesSplit(n_splits=5)

# # 4. Randomized Search setup
# search = RandomizedSearchCV(
#     estimator=pipe,
#     param_distributions=param_distributions,
#     n_iter=25,
#     scoring='neg_log_loss',
#     cv=tscv,
#     random_state=42,
#     n_jobs=-1,
#     error_score='raise'
# )

# search.fit(X_train, y_train)

# # 5. Save best model instance
# best_log_reg_model = search.best_estimator_
# print("--- Logistic Regression Tuning Results ---")
# print(f"Best CV Log Loss: {-search.best_score_:.4f}")
# print("Best Parameters:", search.best_params_)

In [ ]:
# Obsolete: modeling feature importance based on features used in previos model
# import matplotlib.pyplot as plt
# from xgboost import plot_importance

# # 1. Extract the fitted XGBoost model from the VotingClassifier
# # (Assuming you named it 'xgb' when setting up the ensemble)
# xgb_fitted_model = ensemble.named_estimators_['xgb']

# # 2. Plot the feature importance using the extracted model
# plot_importance(xgb_fitted_model, importance_type='weight')
# plt.title('XGBoost Feature Importance')
# plt.show()

In [ ]:
# Obsolete: original ensemble, and also uses invalid calibration X_val, y_val
# from sklearn.ensemble import VotingClassifier
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.frozen import FrozenEstimator
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier

# model_lgb = LGBMClassifier(random_state=42, verbose=-1)
# ensemble = VotingClassifier(
#     estimators=[
#         ('lr', best_log_reg_model),
#         ('xgb', xgb_light),
#         ('lgb', model_lgb)
#     ],
#     voting='soft'
# )

# ensemble.fit(X_train, y_train)

# calibrator = CalibratedClassifierCV(estimator=FrozenEstimator(ensemble), method='isotonic')
# calibrator.fit(X_val, y_val)

In [ ]:
# Obsolete: calibration on original ensemble
# # Improving Log Loss by Probability clipping (prevent absolute 0% or 100%) and  Platt Scaling (fitting logistic regression curve over raw XGBoost outputs)
# # tree-based models like XGBoost makes hard, axis-aligned binary splits that that have predicted probabilities incline to 0 and 1, 
# # logistic curve helps create smooth S-shaped sigmoid function that is more accurate
# ## ^applied on a already optimized model
# import numpy as np
# from sklearn.metrics import log_loss
# from sklearn.calibration import CalibratedClassifierCV


# # # Probability Clipping
# # # Generate raw probabilities from previously tuned model
# # raw_probs = ensemble.predict_proba(X_test)

# # # Clip probabilities to bounds (e.g., 0.001 and 0.999)
# epsilon = 1e-3
# # clipped_probs = np.clip(raw_probs, epsilon, 1 - epsilon)

# # # Evaluate the bounded log loss
# # clipped_loss = log_loss(y_test, clipped_probs) # forces all possibilities to be between


# # Platt Scaling Integration
# # Wrap your base XGBoost estimator and apply Platt scaling (sigmoid)
# # calibration happens on a optimized model
# calibrator = CalibratedClassifierCV(
#     estimator=ensemble, # ensemble model
#     method='sigmoid', 
#     cv=tscv              # Use TimeSeriesSplit (chronological), cuz cv=3 randomly shuffles all rows
# )

# # Fit the calibrator directly on your chronologically sorted training data
# calibrator.fit(X_train, y_train)

# # Predict and evaluate the new calibrated probabilities
# calibrated_probs = calibrator.predict_proba(X_test)

# # clip the calibrated probabilities for maximum safety
# final_probs = np.clip(calibrated_probs, epsilon, 1 - epsilon)
# final_loss = log_loss(y_test, final_probs)

# final_loss = log_loss(y_test, final_probs)
# print(f"Final Calibrated Log Loss: {final_loss:.4f}")

In [ ]:
# Obsolete: predict_match function using old model
# # def predict_match(model, elo_diff, form_diff, is_host, is_neutral):
# #     match_data = pd.DataFrame([{
# #         'elo_difference': elo_diff,
# #         'is_home_team_host': is_host,
# #         'neutral': is_neutral,
# #         'form_difference': form_diff
# #     }]) # 'neutral' meaning neither team's home

# #     # calculate probability (array of array)
# #     probs = model.predict_proba(match_data)[0]

# #     #prints results
# #     print(f"Away Win: {probs[0]*100:.1f}%")
# #     print(f"Draw:     {probs[1]*100:.1f}%")
# #     print(f"Home Win: {probs[2]*100:.1f}%")

# # predict_match(best_model, elo_diff=100, form_diff=-1.0, is_host=True, is_neutral=False)


# # creating match probability based on match data, and using model created to 
# # create probability distribution
# def predict_match(home_team, away_team, is_neutral=True):
#     # Elo val
#     # .get() defaults to 1500 or an empty list if a team has never played before
#     home_elo = elo_tracker.get(home_team, 1500)
#     away_elo = elo_tracker.get(away_team, 1500)
    
#     home_recent_goals = goal_tracker.get(home_team, [])
#     away_recent_goals = goal_tracker.get(away_team, [])
    
#     home_goals_avg = sum(home_recent_goals) / len(home_recent_goals) if len(home_recent_goals) > 0 else 0
#     away_goals_avg = sum(away_recent_goals) / len(away_recent_goals) if len(away_recent_goals) > 0 else 0
    
#     # Calculate the exact features for model 
#     elo_diff = home_elo - away_elo
#     form_diff = home_goals_avg - away_goals_avg
    
#     # If the game is neutral (like a World Cup match), neither advantage
#     is_host = False if is_neutral else True
    
#     # Package it into a DataFrame exactly matching training features list
#     match_data = pd.DataFrame([{
#         'elo_difference': elo_diff,
#         'is_home_team_host': is_host,
#         'neutral': is_neutral,
#         'form_difference': form_diff
#     }])
    
#     # Get probabilities from optimized model
#     # using best_model from gridsearch
#     probs = ensemble.predict_proba(match_data)[0]
    
#     # Display the results cleanly
#     print(f"{'='*40}")
#     print(f" {home_team} vs {away_team} ".center(40, '='))
#     print(f"{'='*40}\n")
#     print(f"📊 Pre-Match Stats:")
#     print(f"  {home_team} Elo: {home_elo:.0f} | Form: {home_goals_avg:.1f} goals/game")
#     print(f"  {away_team} Elo: {away_elo:.0f} | Form: {away_goals_avg:.1f} goals/game\n")
    
#     print(f"🔮 Prediction:")
#     print(f"  {home_team} Win: {probs[2]*100:.1f}%")
#     print(f"  Draw:        {probs[1]*100:.1f}%")
#     print(f"  {away_team} Win: {probs[0]*100:.1f}%")

# predict_match('Japan', 'Australia', is_neutral=True)
# predict_match('Argentina', 'France', is_neutral=True)
# predict_match('Japan', 'Brazil', is_neutral=True)

In [ ]:
# # Testing two combined feature before tuning
# COMBINED_FEATURE_SETS = {
#     "baseline": BASELINE,

#     # Keep the current winner as the comparison.
#     "baseline_plus_h2h": (
#         BASELINE + ADDITIONAL_GROUPS["h2h"]
#     ),

#     "baseline_plus_h2h_scoring": (
#         BASELINE
#         + ADDITIONAL_GROUPS["h2h"]
#         + ADDITIONAL_GROUPS["scoring_level"]
#     ),

#     "baseline_plus_h2h_scoring_rest": (
#         BASELINE
#         + ADDITIONAL_GROUPS["h2h"]
#         + ADDITIONAL_GROUPS["scoring_level"]
#         + ADDITIONAL_GROUPS["rest"]
#     ),
# }

# combined_results = evaluate_feature_sets(
#     development,
#     folds,
#     COMBINED_FEATURE_SETS,
#     {"XGBoost": models["XGBoost"]},
# )

In [ ]:
# Obsolete: inefficient predict_match function
# #Implementing predict_match
# from collections import defaultdict, deque
# import math

# def predict_match(
#     home_team,
#     away_team,
#     match_date,
#     *,
#     is_neutral=True,
#     is_home_team_host=None,
# ):
#     model = selected_model
#     feature_names = list(selected_features)

#     required_features = {
#         "elo_difference",
#         "is_home_team_host",
#         "neutral",
#         "form_difference",
#         "abs_elo_difference",
#     }

#     if set(feature_names) != required_features:
#         raise ValueError(
#             "This function supports the selected LR strength-gap "
#             "feature set. Update the feature builder for another set."
#         )

#     home_team = home_team.strip()
#     away_team = away_team.strip()
#     prediction_date = pd.Timestamp(match_date).normalize()

#     if pd.isna(prediction_date):
#         raise ValueError("Provide a valid match date.")

#     if home_team == away_team:
#         raise ValueError("Select two different teams.")

#     if not isinstance(is_neutral, (bool, np.bool_)):
#         raise ValueError("is_neutral must be True or False.")

#     # Explicitly supply this for non-neutral venues, avoiding an
#     # assumption that the first-listed team must be the actual host.
#     if is_home_team_host is None:
#         if not is_neutral:
#             raise ValueError(
#                 "For a non-neutral match, specify "
#                 "is_home_team_host=True or False."
#             )
#         is_home_team_host = False

#     if not isinstance(is_home_team_host, (bool, np.bool_)):
#         raise ValueError("is_home_team_host must be True or False.")

#     if is_neutral and is_home_team_host:
#         raise ValueError("A neutral match cannot have home-host=True.")

#     # This historical model's calibration used data through this date.
#     model_information_cutoff = pd.Timestamp(
#         calibration["date"].max()
#     ).normalize()

#     if prediction_date <= model_information_cutoff:
#         raise ValueError(
#             "This fitted model uses information through "
#             f"{model_information_cutoff.date()}. "
#             "Use an earlier fitted model for this historical date."
#         )

#     # df is feature-engineered match table.
#     history = df.copy()
#     history["date"] = pd.to_datetime(history["date"])

#     history = (
#         history.loc[
#             (history["date"] < prediction_date)
#             & history["tournament"].ne("Friendly")
#         ]
#         .dropna(subset=["home_score", "away_score"])
#         .sort_values("date", kind="stable")
#     )

#     known_teams = set(history["home_team"]) | set(history["away_team"])
#     unknown = {home_team, away_team} - known_teams

#     if unknown:
#         raise ValueError(
#             "No earlier competitive history for: "
#             + ", ".join(sorted(unknown))
#         )

#     ratings = {}
#     recent_goals = defaultdict(lambda: deque(maxlen=5))
#     last_played = {}

#     # Reproduce the existing notebook's Elo update rule.
#     for row in history.itertuples(index=False):
#         h, a = row.home_team, row.away_team
#         hs, aws = float(row.home_score), float(row.away_score)

#         h_rating = ratings.get(h, 1500.0)
#         a_rating = ratings.get(a, 1500.0)

#         actual_home = 1.0 if hs > aws else 0.0 if hs < aws else 0.5

#         tournament = str(row.tournament)

#         if tournament == "Friendly":
#             k = 20
#         elif tournament == "FIFA World Cup":
#             k = 60
#         elif (
#             "World Cup qualification" in tournament
#             or "UEFA Euro" in tournament
#             or "Copa America" in tournament
#         ):
#             k = 50
#         else:
#             k = 40

#         margin = abs(hs - aws)
#         multiplier = math.sqrt(margin) if margin > 0 else 1.0

#         expected_home = 1 / (
#             1 + 10 ** ((a_rating - h_rating) / 400)
#         )

#         change = k * multiplier * (actual_home - expected_home)

#         ratings[h] = h_rating + change
#         ratings[a] = a_rating - change

#         recent_goals[h].append(hs)
#         recent_goals[a].append(aws)
#         last_played[h] = row.date
#         last_played[a] = row.date

#     home_form = float(np.mean(recent_goals[home_team]))
#     away_form = float(np.mean(recent_goals[away_team]))
#     elo_difference = ratings[home_team] - ratings[away_team]

#     feature_values = {
#         "elo_difference": elo_difference,
#         "is_home_team_host": float(is_home_team_host),
#         "neutral": float(is_neutral),
#         "form_difference": home_form - away_form,
#         "abs_elo_difference": abs(elo_difference),
#     }

#     match_features = pd.DataFrame(
#         [feature_values],
#         columns=feature_names,
#     ).astype(float)

#     probabilities = model.predict_proba(match_features)[0]
#     by_class = dict(zip(model.classes_, probabilities))

#     if (
#         not np.isfinite(probabilities).all()
#         or (probabilities < 0).any()
#         or not np.isclose(probabilities.sum(), 1.0)
#     ):
#         raise ValueError("Model returned invalid probabilities.")

#     return {
#         "home_team": home_team,
#         "away_team": away_team,
#         "match_date": str(prediction_date.date()),
#         "probabilities": {
#             "home_win": float(by_class[2]),
#             "draw": float(by_class[1]),
#             "away_win": float(by_class[0]),
#         },
#         "context": {
#             "home_elo": float(ratings[home_team]),
#             "away_elo": float(ratings[away_team]),
#             "home_recent_goals": home_form,
#             "away_recent_goals": away_form,
#             "home_last_match": str(pd.Timestamp(
#                 last_played[home_team]
#             ).date()),
#             "away_last_match": str(pd.Timestamp(
#                 last_played[away_team]
#             ).date()),
#             "history_cutoff": str(history["date"].max().date()),
#             "model_information_cutoff": str(
#                 model_information_cutoff.date()
#             ),
#         },
#     }

In [ ]:
# # Loading old checkpoint
# import joblib
# model = PROJECT_ROOT / "checkpoints" / "v1_20260915_113112_401128" / "model_bundle.joblib"
# bundle = joblib.load(
#     model
# )

# selected_model = bundle["model"]
# selected_features = bundle["features"]
# selected_name = bundle["model_name"]

In [ ]:
# import joblib
# # Loading the saved checkpoints/artifacts
# prediction_bundle = joblib.load(
#     checkpoint_dir / "model_bundle.joblib"
# )

# prediction_state = joblib.load(
#     checkpoint_dir / "latest_team_state.joblib"
# )

# assert (
#     prediction_bundle["feature_version"]
#     == prediction_state["feature_version"]
# )

# assert prediction_state["checkpoint_id"] == checkpoint_dir.name

In [ ]:
# Metadata-cleanup code, after cleaning 2 bundles, obsolete
from pathlib import Path, PureWindowsPath
import joblib

project_root = Path.cwd()

assert (project_root / "checkpoints").is_dir()
assert (project_root / "artifacts").is_dir()

bundle_paths = list(
    (project_root / "checkpoints").rglob("model_bundle.joblib")
)
bundle_paths.append(
    project_root / "artifacts" / "model_bundle.joblib"
)

changed = 0

for bundle_path in bundle_paths:
    bundle = joblib.load(bundle_path)
    source = bundle.get("source_checkpoint")

    if isinstance(source, str):
        cleaned = PureWindowsPath(source).name

        if cleaned != source:
            bundle["source_checkpoint"] = cleaned
            joblib.dump(bundle, bundle_path)
            changed += 1

# Verify that this metadata field no longer contains directory paths.
for bundle_path in bundle_paths:
    source = joblib.load(bundle_path).get("source_checkpoint")
    if isinstance(source, str):
        assert source == PureWindowsPath(source).name

print(f"Cleaned {changed} bundles. Provenance metadata verified.")